In [ ]:
import warnings; warnings.filterwarnings('ignore')
import re, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import gensim, gensim.corpora

with open('stopwords_english') as f:
    stopwords = set(f.read().splitlines())

df = pd.read_csv('test.csv', header=None, names=['label','title','content']).dropna(subset=['content'])

def clean(text):
    s = str(text).lower()
    s = re.sub(r'\d+', '', s)
    s = re.sub(r'[^\w\s]', '', s)
    toks = [t for t in s.split() if t.isalpha() and t not in stopwords and len(t) > 2]
    return toks

docs = [clean(t) for t in df['content']]
docs = [d for d in docs if d]

In [ ]:
D1 = gensim.corpora.Dictionary(docs)
D1.filter_extremes(no_below=2)

matrix_of_doc_term = [D1.doc2bow(doc) for doc in docs]

ldamodel = gensim.models.ldamodel.LdaModel(
    matrix_of_doc_term, num_topics=10, id2word=D1, passes=100, random_state=17
)

for i in range(10):
    print(f'topic {i}:', ', '.join([w for w,_ in ldamodel.show_topic(i, topn=5)]))

In [ ]:
top5 = ldamodel.show_topic(0, topn=5)
words = [w for w, _ in top5]
probs = [p for _, p in top5]

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(words[::-1], probs[::-1], color='steelblue')
ax.set_xlabel('probability'); ax.set_title('top 5 terms - topic 0')
plt.tight_layout(); plt.show()

In [ ]:
kl_matrix, _ = ldamodel.diff(ldamodel, distance='kullback_leibler', num_words=50)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(kl_matrix, cmap='YlOrRd')
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xticklabels([f'T{i}' for i in range(10)])
ax.set_yticklabels([f'T{i}' for i in range(10)])
ax.set_title('KL divergence across 10 LDA topics (num_words=50)')
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()